# Train Personalized DeepVQE voi 1/10 dataset tren Google Colab

Notebook nay tao subset bang 10% `metadata.json`, chia train/valid trong subset do, roi chay `train_personalized.py` de danh gia nhanh pipeline va checkpoint.

## 1. Kiem tra GPU va mount Google Drive

In [ ]:
import os
import sys
from pathlib import Path

import torch

print(f"Python: {sys.version}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from google.colab import drive
drive.mount('/content/drive')

## 2. Cai thu vien

In [ ]:
!pip install -q einops==0.7.0 speechbrain soundfile tqdm

import torchaudio
print(f"Torchaudio: {torchaudio.__version__}")

## 3. Tro toi source code va dataset

In [ ]:
# Sua duong dan nay neu project deepvqe nam o vi tri khac trong Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/deepvqe')

# Dataset mac dinh khop voi data_prep_colab.ipynb.
DATASET_DIR = Path('/content/drive/MyDrive/TSE_Dataset/mixed_dataset')
METADATA_PATH = DATASET_DIR / 'metadata.json'

# Neu muon clone tu GitHub thay vi dung Drive, dien URL roi bat USE_GIT_CLONE=True.
USE_GIT_CLONE = False
GIT_REPO_URL = ''

if USE_GIT_CLONE:
    assert GIT_REPO_URL, 'Hay dien GIT_REPO_URL truoc khi clone.'
    clone_dir = Path('/content/deepvqe')
    if not clone_dir.exists():
        !git clone {GIT_REPO_URL} {clone_dir}
    PROJECT_DIR = clone_dir

assert (PROJECT_DIR / 'train_personalized.py').exists(), f'Khong thay train_personalized.py trong {PROJECT_DIR}'
assert METADATA_PATH.exists(), f'Khong thay metadata.json tai {METADATA_PATH}. Hay chay data_prep_colab.ipynb truoc.'

os.chdir(PROJECT_DIR)
print(f'Project: {PROJECT_DIR}')
print(f'Dataset: {DATASET_DIR}')

## 4. Tao manifest subset 10%

In [ ]:
import json
import random
from collections import Counter

SEED = 1337
SUBSET_RATIO = 0.10
VALID_RATIO = 0.10

# Muc 2.1 roadmap: 60% full TSE, 15% denoise, 15% separation, 10% absent speaker.
CASE_RATIOS = {
    'full_tse': 0.60,
    'target_noise': 0.15,
    'target_interferer': 0.15,
    'absent_speaker': 0.10,
}

# Thu muc rieng de khong ghi de manifest full dataset.
MANIFEST_DIR = DATASET_DIR / 'manifests_10pct_eval'
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

with METADATA_PATH.open('r', encoding='utf-8') as f:
    all_records = json.load(f)

assert isinstance(all_records, list) and all_records, 'metadata.json phai la list va khong duoc rong.'

rng = random.Random(SEED)

def normalized_case(item):
    case = str(item.get('case', 'unknown')).lower()
    if bool(item.get('is_negative', False)) or case in {'negative', 'absent', 'absent_speaker', 'wrong_speaker'}:
        return 'absent_speaker'
    return case

def allocate_counts(total, ratios):
    raw = {case: total * ratio for case, ratio in ratios.items()}
    counts = {case: int(value) for case, value in raw.items()}
    while sum(counts.values()) < total:
        case = max(ratios, key=lambda key: raw[key] - counts[key])
        counts[case] += 1
    while sum(counts.values()) > total:
        case = max(counts, key=counts.get)
        counts[case] -= 1
    return counts

def group_by_case(items):
    groups = {case: [] for case in CASE_RATIOS}
    extras = []
    for item in items:
        case = normalized_case(item)
        if case in groups:
            groups[case].append(item)
        else:
            extras.append(item)
    return groups, extras

subset_size = min(len(all_records), max(len(CASE_RATIOS), int(len(all_records) * SUBSET_RATIO)))
target_counts = allocate_counts(subset_size, CASE_RATIOS)
groups, extras = group_by_case(all_records)

subset_records = []
remaining = []
shortage = 0
for case, target_count in target_counts.items():
    items = list(groups[case])
    rng.shuffle(items)
    take = min(target_count, len(items))
    subset_records.extend(items[:take])
    remaining.extend(items[take:])
    if take < target_count:
        shortage += target_count - take
        print(f'Warning: case {case} needs {target_count}, only has {take}.')

remaining.extend(extras)
rng.shuffle(remaining)
if len(subset_records) < subset_size:
    subset_records.extend(remaining[:subset_size - len(subset_records)])

subset_records = subset_records[:subset_size]
rng.shuffle(subset_records)

selected_groups, selected_extras = group_by_case(subset_records)
train_records = []
valid_records = []
for case in CASE_RATIOS:
    items = list(selected_groups[case])
    rng.shuffle(items)
    valid_count = int(round(len(items) * VALID_RATIO))
    if len(items) > 1 and VALID_RATIO > 0:
        valid_count = max(1, min(len(items) - 1, valid_count))
    valid_records.extend(items[:valid_count])
    train_records.extend(items[valid_count:])

rng.shuffle(selected_extras)
extra_valid_count = int(round(len(selected_extras) * VALID_RATIO))
valid_records.extend(selected_extras[:extra_valid_count])
train_records.extend(selected_extras[extra_valid_count:])
rng.shuffle(train_records)
rng.shuffle(valid_records)

assert train_records and valid_records, 'Subset qua nho, hay tang SUBSET_RATIO hoac tao them data.'

subset_manifest = MANIFEST_DIR / 'subset_10pct.jsonl'
train_manifest = MANIFEST_DIR / 'train_10pct.jsonl'
valid_manifest = MANIFEST_DIR / 'valid_10pct.jsonl'

def write_jsonl(path, items):
    with path.open('w', encoding='utf-8') as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

def summarize(name, items):
    cases = Counter(normalized_case(item) for item in items)
    negatives = cases.get('absent_speaker', 0)
    print(f'{name}: {len(items)} samples, negative={negatives} ({negatives / max(1, len(items)):.1%})')
    print(f'{name} cases: {dict(cases)}')

write_jsonl(subset_manifest, subset_records)
write_jsonl(train_manifest, train_records)
write_jsonl(valid_manifest, valid_records)

print(f'Total metadata: {len(all_records)}')
print(f'Target subset counts: {target_counts}')
print(f'Subset 10%: {len(subset_records)} -> {subset_manifest}')
print(f'Train: {len(train_records)} -> {train_manifest}')
print(f'Valid: {len(valid_records)} -> {valid_manifest}')
summarize('subset', subset_records)
summarize('train', train_records)
summarize('valid', valid_records)
print('Example record:')
print(json.dumps(train_records[0], ensure_ascii=False, indent=2))

## 5. Cau hinh train nhanh de danh gia

In [ ]:
# phase1_reconstruction: nen dung cho lan danh gia dau tien.
PHASE = 'phase1_reconstruction'

# Tang EPOCHS neu muon so sanh chat luong nghiem tuc hon tren subset 10%.
EPOCHS = 5
BATCH_SIZE = 2
NUM_WORKERS = 2

# True: dung enrollment wav va ECAPA online, chay duoc ngay voi metadata tu data_prep_colab.
USE_ONLINE_ECAPA = True

# Thu muc rieng cho checkpoint subset 10%.
RUN_ROOT = Path('/content/drive/MyDrive/TSE_Dataset/runs_10pct_eval')
OUTPUT_DIR = RUN_ROOT / PHASE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Tu resume tu OUTPUT_DIR / 'last.pt' neu Colab bi ngat.
AUTO_RESUME = True

# Neu muon resume thu cong tu checkpoint khac, dien path tai day.
RESUME_FROM = None

print(f'Phase: {PHASE}')
print(f'Output: {OUTPUT_DIR}')

## 6. Chay train

In [ ]:
import subprocess

device = 'cuda' if torch.cuda.is_available() else 'cpu'

cmd = [
    sys.executable,
    'train_personalized.py',
    '--phase', PHASE,
    '--train-manifest', str(train_manifest),
    '--valid-manifest', str(valid_manifest),
    '--data-root', str(DATASET_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--device', device,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
]

if USE_ONLINE_ECAPA:
    cmd.append('--online-ecapa')

if AUTO_RESUME:
    cmd.append('--auto-resume')

if RESUME_FROM:
    cmd.extend(['--resume', str(RESUME_FROM)])

print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 7. Kiem tra checkpoint va ket qua nhanh

In [ ]:
import json

for name in ['best.pt', 'last.pt', 'config.json']:
    path = OUTPUT_DIR / name
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f'{name}: {path} ({size_mb:.2f} MB)')
    else:
        print(f'{name}: chua co')

last_ckpt = OUTPUT_DIR / 'last.pt'
if last_ckpt.exists():
    ckpt = torch.load(str(last_ckpt), map_location='cpu')
    print(f"last.pt epoch={ckpt.get('epoch')} best_metric={ckpt.get('best_metric')} bad_epochs={ckpt.get('bad_epochs')}")

config_path = OUTPUT_DIR / 'config.json'
if config_path.exists():
    with config_path.open('r', encoding='utf-8') as f:
        cfg = json.load(f)
    print('train_manifest:', cfg['data']['train_manifest'])
    print('valid_manifest:', cfg['data']['valid_manifest'])